# Phase 3b — MedSAM + LoRA
**Project:** Can LoRA-Adapted SAM Match Specialist Polyp Networks?

MedSAM (Ma et al. 2024) is SAM ViT-B fine-tuned on SA-Med2D-20M — 20 million medical images
across modalities. Same LoRA + decoder architecture as notebook 03, different starting weights.
Runs on a free T4 (~375 MB checkpoint).

## 1. Setup

In [ ]:
import sys, os
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    REPO = '/content/msu2026summer_final_project'
    if not os.path.exists(REPO):
        !git clone --quiet https://github.com/palism1/msu2026summer_final_project.git {REPO}
    %cd {REPO}
    !git fetch --quiet origin && git reset --hard origin/main
    !find . -name '__pycache__' -type d -exec rm -rf {} + 2>/dev/null; true
    !pip install -q -r requirements.txt
    !pip install -q git+https://github.com/facebookresearch/segment-anything.git

repo_root = os.getcwd()
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

import gdown, shutil, zipfile
from pathlib import Path

DATA_ROOT = Path('data/polyp')
DATA_ROOT.mkdir(parents=True, exist_ok=True)

def _dl(file_id, zip_name):
    target = DATA_ROOT / zip_name.replace('.zip', '')
    if target.exists():
        print(f'{target.name}: already present'); return
    zip_path = f'/tmp/{zip_name}'
    print(f'Downloading {target.name}...')
    gdown.download(id=file_id, output=zip_path, quiet=False, fuzzy=True)
    tmp = Path(f'/tmp/extract_{target.name}')
    tmp.mkdir(exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as zf:
        zf.extractall(tmp)
    extracted = [p for p in tmp.iterdir() if not p.name.startswith('__')]
    if len(extracted) == 1 and extracted[0].is_dir():
        shutil.move(str(extracted[0]), str(target))
    else:
        target.mkdir(exist_ok=True)
        for p in extracted: shutil.move(str(p), str(target / p.name))
    print(f'Done -> {target}')

_dl('1YiGHLw4iTvKdvbT6MgwO9zcCv8zJ_Bnb', 'TrainDataset.zip')
_dl('1Y2z7FD5p5y31vkZwQQomXFRB0HutHyao', 'TestDataset.zip')

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}  |  GPU: {torch.cuda.get_device_name(0) if device=="cuda" else "N/A"}')

## 2. Download MedSAM Checkpoint (~375 MB)

In [ ]:
MEDSAM_FNAME = 'medsam_vit_b.pth'

# Remove any previously corrupted/partial download
if os.path.exists(MEDSAM_FNAME) and os.path.getsize(MEDSAM_FNAME) < 100_000_000:
    print(f'Removing incomplete file ({os.path.getsize(MEDSAM_FNAME)} bytes)')
    os.remove(MEDSAM_FNAME)

if not os.path.exists(MEDSAM_FNAME):
    from huggingface_hub import hf_hub_download, list_repo_files
    REPO_ID = 'bowang-lab/MedSAM'
    try:
        downloaded = hf_hub_download(repo_id=REPO_ID, filename='medsam_vit_b.pth')
    except Exception as e:
        print(f'Could not fetch medsam_vit_b.pth: {e}')
        print('Files actually available in the repo:')
        for f in list_repo_files(REPO_ID):
            print(' ', f)
        raise
    import shutil
    shutil.copy(downloaded, MEDSAM_FNAME)
    print(f'Downloaded -> {MEDSAM_FNAME} ({os.path.getsize(MEDSAM_FNAME)/1e6:.1f} MB)')
else:
    print(f'Checkpoint ready: {MEDSAM_FNAME} ({os.path.getsize(MEDSAM_FNAME)/1e6:.1f} MB)')

## 3. Build MedSAM-LoRA Model

In [ ]:
from src.models import build_medsam_lora

LORA_R     = 4
LORA_ALPHA = 8.0
IMG_SIZE   = 352

model = build_medsam_lora(
    medsam_checkpoint=MEDSAM_FNAME,
    lora_r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.1,
    img_size=IMG_SIZE, device=device,
)
model = model.to(device)

total     = model.total_parameters()
trainable = model.trainable_parameters()
print(f'MedSAM ViT-B + LoRA (r={LORA_R})')
print(f'  Total parameters:     {total:>12,}')
print(f'  Trainable parameters: {trainable:>12,}  ({100*trainable/total:.2f}%)')
print(f'  Frozen parameters:    {total-trainable:>12,}')

In [ ]:
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE).to(device)
with torch.no_grad():
    out = model(dummy)
print(f'Forward pass OK: {list(dummy.shape)} -> {list(out.shape)}')

## 4. Parameter Efficiency Comparison

In [ ]:
from src.models import build_unet
from src.models.unet import count_parameters

unet_p = count_parameters(build_unet())

print(f'{"Method":<32} {"Total params":>13}  {"Trainable":>13}  {"% trainable":>12}')
print('-' * 76)
print(f'{"U-Net (ResNet-34)":<32} {unet_p["total"]:>13,}  {unet_p["trainable"]:>13,}  {100*unet_p["trainable"]/unet_p["total"]:>11.1f}%')
print(f'{"MedSAM ViT-B + LoRA (r=4)":<32} {total:>13,}  {trainable:>13,}  {100*trainable/total:>11.2f}%')

## 5. Training

In [ ]:
from pathlib import Path
from torch.utils.data import DataLoader
from src.data import PolypDataset, build_splits, get_train_transform, get_val_transform
from src.metrics import MetricTracker
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm.notebook import tqdm
import numpy as np

DATA_ROOT = Path('data/polyp')
SEED      = 42
BATCH     = 4
EPOCHS    = 50
LR        = 5e-5
PATIENCE  = 8

splits = build_splits(DATA_ROOT, seed=SEED)

train_ds = PolypDataset(
    splits['train']['image_paths'], splits['train']['mask_paths'],
    transform=get_train_transform(IMG_SIZE))
val_ds = PolypDataset(
    splits['seen_kvasir']['image_paths'] + splits['seen_clinicdb']['image_paths'],
    splits['seen_kvasir']['mask_paths']  + splits['seen_clinicdb']['mask_paths'],
    transform=get_val_transform(IMG_SIZE))

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

def dice_bce_loss(logits, targets, eps=1e-6):
    probs = torch.sigmoid(logits)
    bce = nn.functional.binary_cross_entropy_with_logits(logits, targets)
    inter = (probs * targets).sum(dim=(1,2,3))
    dice = 1 - (2*inter + eps) / (probs.sum(dim=(1,2,3)) + targets.sum(dim=(1,2,3)) + eps)
    return bce + dice.mean()

optimizer = AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LR, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS)
CKPT_DIR  = Path(f'checkpoints/medsam/seed{SEED}')
CKPT_DIR.mkdir(parents=True, exist_ok=True)

best_dice, patience_count = 0.0, 0
history = {'train_loss': [], 'val_dice': [], 'val_iou': []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    for imgs, masks in tqdm(train_loader, desc=f'Epoch {epoch:03d} train', leave=False):
        imgs, masks = imgs.to(device), masks.to(device)
        optimizer.zero_grad()
        loss = dice_bce_loss(model(imgs), masks)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    epoch_loss /= len(train_loader)

    model.eval()
    tracker = MetricTracker()
    with torch.no_grad():
        for imgs, masks in val_loader:
            probs = torch.sigmoid(model(imgs.to(device))).cpu().numpy()
            for i in range(len(probs)):
                tracker.update(probs[i, 0], masks[i, 0].numpy())
    scores = tracker.compute()
    scheduler.step()

    history['train_loss'].append(epoch_loss)
    history['val_dice'].append(scores['dice'])
    history['val_iou'].append(scores['iou'])

    print(f'Epoch {epoch:03d} | loss={epoch_loss:.4f} | val_dice={scores["dice"]:.4f} | val_iou={scores["iou"]:.4f}')

    if scores['dice'] > best_dice:
        best_dice = scores['dice']
        patience_count = 0
        torch.save(model.state_dict(), CKPT_DIR / 'best.pt')
        print(f'  New best Dice: {best_dice:.4f}')
    else:
        patience_count += 1
        if patience_count >= PATIENCE:
            print(f'Early stopping at epoch {epoch}')
            break

print(f'\nBest val Dice: {best_dice:.4f}')

## 6. Full Evaluation — Seen + Unseen (Generalization Study)

In [ ]:
model.load_state_dict(torch.load(CKPT_DIR / 'best.pt', map_location=device))
model.eval()
val_transform = get_val_transform(IMG_SIZE)

split_labels = {
    'Seen — Kvasir':        'seen_kvasir',
    'Seen — CVC-ClinicDB':  'seen_clinicdb',
    'Unseen — CVC-ColonDB': 'cvc_colondb',
    'Unseen — ETIS-Larib':  'etis_larib',
    'Unseen — CVC-300':     'cvc_300',
}

print(f'MedSAM ViT-B + LoRA (r={LORA_R})')
print(f'{"Split":<26} {"mDice":>7} {"mIoU":>7} {"MAE":>7} {"wFm":>7} {"Sm":>7} {"Em":>7}')
print('-' * 75)

results = {}
for label, key in split_labels.items():
    if key not in splits:
        print(f'{label:<26} -- not in splits --')
        continue
    ds = PolypDataset(splits[key]['image_paths'], splits[key]['mask_paths'], transform=val_transform)
    loader = DataLoader(ds, batch_size=4, shuffle=False, num_workers=2)
    tracker = MetricTracker()
    with torch.no_grad():
        for imgs, masks in loader:
            probs = torch.sigmoid(model(imgs.to(device))).cpu().numpy()
            for i in range(len(probs)):
                tracker.update(probs[i, 0], masks[i, 0].numpy())
    sc = tracker.compute()
    results[key] = sc
    print(f'{label:<26} {sc["dice"]:>7.4f} {sc["iou"]:>7.4f} {sc["mae"]:>7.4f} '
          f'{sc["wfm"]:>7.4f} {sc["sm"]:>7.4f} {sc["em"]:>7.4f}')

## Phase 3b Summary

| Deliverable | Status |
|---|---|
| MedSAM ViT-B checkpoint loaded | ✅ |
| LoRA injected (same config as SAM, notebook 03) | ✅ |
| Training (identical protocol to U-Net and SAM) | ✅ |
| Full seen/unseen evaluation | ✅ |

**Next:** `05_benchmark.ipynb` (Phase 4) — side-by-side comparison of all three models.